In [1]:
!pip install tensorflow --user


^C


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [2]:
!pip install --upgrade tensorflow

In [1]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install mediapipe opencv-python

In [3]:
!pip install keras

In [4]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [5]:
!pip install sklearn matplotlib

  Created wheel for sklearn: filename=sklearn-0.0-py2.py3-none-any.whl size=1309 sha256=b9811b0024ac3bebf573090fc0981e29543c88707acc80599235f7c0654b3305
  Stored in directory: c:\users\vinay_chanamallu\appdata\local\pip\cache\wheels\e4\7b\98\b6466d71b8d738a0c547008b9eb39bf8676d1ff6ca4b22af1c
Successfully built sklearn


In [88]:
import cv2
import numpy as np
import os
#from matplotlib import pyplot as plt
import time
import mediapipe as mp

In [89]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [90]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [91]:
def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), 
                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                             ) 
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [92]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

In [96]:
# Path for exported data, numpy arrays
path="/x/y/z"
DATA_PATH = os.path.join(path,'MP_Data') 
===>/x/y/z/MP_DATA
# Actions that we try to detect
actions = np.array(['standing or walking', 'sitting', 'fell_alarm'])

# Thirty videos worth of data
no_sequences = 30

# Videos are going to be 30 frames in length
sequence_length = 30

# Folder start
start_folder = 30

In [97]:
for action in actions: 
    for sequence in range(no_sequences):
        try: 
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

In [99]:
cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.3, min_tracking_confidence=0.3) as holistic:
    
    # NEW LOOP
    # Loop through actions
    for action in actions:
        # Loop through sequences aka videos
        for sequence in range(no_sequences):
            # Loop through video length aka sequence length
            for frame_num in range(sequence_length):

                # Read feed
                ret, frame = cap.read()

                # Make detections
                image, results = mediapipe_detection(frame, holistic)
#                 print(results)

                # Draw landmarks
                draw_styled_landmarks(image, results)
                
                # NEW Apply wait logic
                if frame_num == 0: 
                    cv2.putText(image, 'STARTING COLLECTION', (120,200), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(6000)
                else: 
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                
                # NEW Export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break
                    
    cap.release()
    cv2.destroyAllWindows()

In [100]:
from sklearn.model_selection import train_test_split
#from keras.utils import to_categorical
# from keras.utils import np_utils
#import keras.utils.np_utils.to_categorical
from tensorflow.keras.utils import to_categorical


In [101]:
label_map = {label:num for num, label in enumerate(actions)}

In [102]:
print(label_map)

{'standing or walking': 0, 'sitting': 1, 'fell_alarm': 2}


In [103]:
sequences, labels = [], []
for action in actions:
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [104]:
np.array(labels).shape

(90,)

In [105]:

np.array(sequences).shape

(90, 30, 1662)

In [106]:
X = np.array(sequences)

In [107]:
y = to_categorical(labels).astype(int)

In [108]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

In [109]:
X_train.shape

(85, 30, 1662)

In [110]:
y_train.shape

(85, 3)

In [111]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [112]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [113]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30,1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
#model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

In [114]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [127]:
model.fit(X_train, y_train, epochs=300, callbacks=[tb_callback])

Epoch 1/300
3/3 [==============================] - 0s 71ms/step - loss: 0.1910 - categorical_accuracy: 0.9529
Epoch 2/300
3/3 [==============================] - 0s 59ms/step - loss: 0.1681 - categorical_accuracy: 0.9647
Epoch 3/300
3/3 [==============================] - 0s 62ms/step - loss: 0.1541 - categorical_accuracy: 0.9647
Epoch 4/300
3/3 [==============================] - 0s 61ms/step - loss: 0.1402 - categorical_accuracy: 0.9765
Epoch 5/300
3/3 [==============================] - 0s 69ms/step - loss: 0.1328 - categorical_accuracy: 0.9765
Epoch 6/300
3/3 [==============================] - 0s 67ms/step - loss: 0.1268 - categorical_accuracy: 0.9765
Epoch 7/300
3/3 [==============================] - 0s 66ms/step - loss: 0.1194 - categorical_accuracy: 0.9765
Epoch 8/300
3/3 [==============================] - 0s 67ms/step - loss: 0.1115 - categorical_accuracy: 0.9765
Epoch 9/300
3/3 [==============================] - 0s 66ms/step - loss: 0.1079 - categorical_accuracy: 0.9765
Epoch 10/3

3/3 [==============================] - 0s 64ms/step - loss: 0.0024 - categorical_accuracy: 1.0000
Epoch 75/300
3/3 [==============================] - 0s 66ms/step - loss: 0.0023 - categorical_accuracy: 1.0000
Epoch 76/300
3/3 [==============================] - 0s 64ms/step - loss: 0.0022 - categorical_accuracy: 1.0000
Epoch 77/300
3/3 [==============================] - 0s 64ms/step - loss: 0.0022 - categorical_accuracy: 1.0000
Epoch 78/300
3/3 [==============================] - 0s 68ms/step - loss: 0.0022 - categorical_accuracy: 1.0000
Epoch 79/300
3/3 [==============================] - 0s 66ms/step - loss: 0.0021 - categorical_accuracy: 1.0000
Epoch 80/300
3/3 [==============================] - 0s 62ms/step - loss: 0.0020 - categorical_accuracy: 1.0000
Epoch 81/300
3/3 [==============================] - 0s 60ms/step - loss: 0.0020 - categorical_accuracy: 1.0000
Epoch 82/300
3/3 [==============================] - 0s 61ms/step - loss: 0.0020 - categorical_accuracy: 1.0000
Epoch 83/300
3

3/3 [==============================] - 0s 90ms/step - loss: 6.3843e-04 - categorical_accuracy: 1.0000
Epoch 147/300
3/3 [==============================] - 0s 94ms/step - loss: 6.3545e-04 - categorical_accuracy: 1.0000
Epoch 148/300
3/3 [==============================] - 0s 94ms/step - loss: 6.1905e-04 - categorical_accuracy: 1.0000
Epoch 149/300
3/3 [==============================] - 0s 95ms/step - loss: 6.1637e-04 - categorical_accuracy: 1.0000
Epoch 150/300
3/3 [==============================] - 0s 93ms/step - loss: 6.0972e-04 - categorical_accuracy: 1.0000
Epoch 151/300
3/3 [==============================] - 0s 97ms/step - loss: 6.0385e-04 - categorical_accuracy: 1.0000
Epoch 152/300
3/3 [==============================] - 0s 90ms/step - loss: 5.9576e-04 - categorical_accuracy: 1.0000
Epoch 153/300
3/3 [==============================] - 0s 86ms/step - loss: 5.8810e-04 - categorical_accuracy: 1.0000
Epoch 154/300
3/3 [==============================] - 0s 84ms/step - loss: 5.7954e-04 -

3/3 [==============================] - 0s 83ms/step - loss: 5.0332e-04 - categorical_accuracy: 1.0000
Epoch 217/300
3/3 [==============================] - 0s 85ms/step - loss: 4.7587e-04 - categorical_accuracy: 1.0000
Epoch 218/300
3/3 [==============================] - 0s 86ms/step - loss: 4.7284e-04 - categorical_accuracy: 1.0000
Epoch 219/300
3/3 [==============================] - 0s 88ms/step - loss: 4.5314e-04 - categorical_accuracy: 1.0000
Epoch 220/300
3/3 [==============================] - 0s 86ms/step - loss: 4.3667e-04 - categorical_accuracy: 1.0000
Epoch 221/300
3/3 [==============================] - 0s 84ms/step - loss: 4.2476e-04 - categorical_accuracy: 1.0000
Epoch 222/300
3/3 [==============================] - 0s 84ms/step - loss: 4.0523e-04 - categorical_accuracy: 1.0000
Epoch 223/300
3/3 [==============================] - 0s 81ms/step - loss: 3.9189e-04 - categorical_accuracy: 1.0000
Epoch 224/300
3/3 [==============================] - 0s 76ms/step - loss: 3.7960e-04 -

3/3 [==============================] - 0s 82ms/step - loss: 2.0917e-04 - categorical_accuracy: 1.0000
Epoch 287/300
3/3 [==============================] - 0s 80ms/step - loss: 2.0824e-04 - categorical_accuracy: 1.0000
Epoch 288/300
3/3 [==============================] - 0s 79ms/step - loss: 2.0671e-04 - categorical_accuracy: 1.0000
Epoch 289/300
3/3 [==============================] - 0s 81ms/step - loss: 2.0533e-04 - categorical_accuracy: 1.0000
Epoch 290/300
3/3 [==============================] - 0s 80ms/step - loss: 2.0388e-04 - categorical_accuracy: 1.0000
Epoch 291/300
3/3 [==============================] - 0s 79ms/step - loss: 2.0242e-04 - categorical_accuracy: 1.0000
Epoch 292/300
3/3 [==============================] - 0s 83ms/step - loss: 2.0118e-04 - categorical_accuracy: 1.0000
Epoch 293/300
3/3 [==============================] - 0s 81ms/step - loss: 1.9983e-04 - categorical_accuracy: 1.0000
Epoch 294/300
3/3 [==============================] - 0s 81ms/step - loss: 1.9852e-04 -

In [128]:
res = model.predict(X_test)

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [129]:
actions[np.argmax(res[4])]

'sitting'

In [130]:
actions[np.argmax(y_test[4])]

'sitting'

In [131]:
model.save('actionsuper.h5')

In [120]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)
        
    return output_frame

In [133]:
# 1. New detection variables
sequence = []
sentence = []
threshold = 1

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        
        # 2. Prediction logic
        keypoints = extract_keypoints(results)
#         sequence.insert(0,keypoints)
#         sequence = sequence[:30]
        sequence.append(keypoints)
        sequence = sequence[-30:]
        
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(actions[np.argmax(res)])
            
            
        #3. Viz logic
            if res[np.argmax(res)] > threshold: 
                if len(sentence) > 0: 
                    if actions[np.argmax(res)] != sentence[-1]:
                        sentence.append(actions[np.argmax(res)])
                else:
                    sentence.append(actions[np.argmax(res)])

#             if len(sentence) > 5: 
#                 sentence = sentence[-5:]

            # Viz probabilities
            image = prob_viz(res, actions, image, colors)
            
#         cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
#        # cv2.putText(image, ' '.join(sentence), (3,30), 
#                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe.python.solution_base.SolutionOutputs'>
sitting
<class 'mediapipe